# Hands-on 5 — SPARQL queries

Une cellule d’installation/chargement, puis **une cellule par requête** (Q1 à Q7). Les requêtes sont reprises telles quelles du fichier fourni.

In [1]:
# Si nécessaire, décommenter la ligne suivante :
# %pip install rdflib pandas

from rdflib import Graph
import pandas as pd

g = Graph()
g.parse("knowledge-graph-with-links.ttl", format="turtle")
print(f"Graphe chargé : {len(g)} triplets")

Graphe chargé : 16527 triplets


## Q1 - Inspect links to Wikidata for each linkable class.

In [9]:
query = """
PREFIX db: <https://example.org/db-delays/ontology/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?class (COUNT(DISTINCT ?resource) AS ?linkedResources)
       (COUNT(DISTINCT ?wikidata) AS ?wikidataEntities)
WHERE {
  VALUES ?class { db:Station db:City db:State }
  ?resource a ?class ; owl:sameAs ?wikidata .
}
GROUP BY ?class
ORDER BY ?class
"""

results = g.query(query)
display(pd.DataFrame([[str(v) if v is not None else None for v in row] for row in results],
                     columns=[str(v) for v in results.vars]))

,class,linkedResources,wikidataEntities
0,https://example.org/db-delays/ontology/#City,167,167
1,https://example.org/db-delays/ontology/#State,16,16
2,https://example.org/db-delays/ontology/#Station,251,250


## Q2 - Inspect the Station -> City -> State chain and its external links.

In [3]:
query = """
PREFIX db: <https://example.org/db-delays/ontology/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?station ?stationName ?stationWikidata
                ?cityName ?cityWikidata ?stateName ?stateWikidata
WHERE {
  ?station a db:Station ;
           rdfs:label ?stationName ;
           db:inCity ?city ;
           owl:sameAs ?stationWikidata .
  ?city a db:City ; rdfs:label ?cityName ;
        db:inState ?state ; owl:sameAs ?cityWikidata .
  ?state a db:State ; rdfs:label ?stateName ; owl:sameAs ?stateWikidata .
}
ORDER BY ?stateName ?cityName ?stationName
"""

results = g.query(query)
display(pd.DataFrame([[str(v) if v is not None else None for v in row] for row in results],
                     columns=[str(v) for v in results.vars]))

,station,stationName,stationWikidata,cityName,cityWikidata,stateName,stateWikidata
0,https://example.org/db-delays/resource/Station...,Allensbach station,https://wikidata.org/entity/Q11685458,Allensbach,http://www.wikidata.org/entity/Q61939,Baden-Württemberg,https://wikidata.org/entity/Q985
1,https://example.org/db-delays/resource/Station...,Altbach,https://wikidata.org/entity/Q18681700,Altbach,http://www.wikidata.org/entity/Q80539,Baden-Württemberg,https://wikidata.org/entity/Q985
2,https://example.org/db-delays/resource/Station...,Asperg station,https://wikidata.org/entity/Q46550,Asperg,http://www.wikidata.org/entity/Q61621,Baden-Württemberg,https://wikidata.org/entity/Q985
3,https://example.org/db-delays/resource/Station...,Auggen railway stop,https://wikidata.org/entity/Q2106629,Auggen,http://www.wikidata.org/entity/Q268734,Baden-Württemberg,https://wikidata.org/entity/Q985
4,https://example.org/db-delays/resource/Station...,Aulendorf station,https://wikidata.org/entity/Q165899,Aulendorf,http://www.wikidata.org/entity/Q505459,Baden-Württemberg,https://wikidata.org/entity/Q985
...,...,...,...,...,...,...,...
246,https://example.org/db-delays/resource/Station...,Bredstedt station,https://wikidata.org/entity/Q109380180,Bredstedt,http://www.wikidata.org/entity/Q20753,Schleswig-Holstein,https://wikidata.org/entity/Q1194
247,https://example.org/db-delays/resource/Station...,Büchen railway station,https://wikidata.org/entity/Q800600,Büchen,http://www.wikidata.org/entity/Q631683,Schleswig-Holstein,https://wikidata.org/entity/Q1194
248,https://example.org/db-delays/resource/Station...,Altenburg railway station,https://wikidata.org/entity/Q318061,Altenburg,http://www.wikidata.org/entity/Q15976,Thuringia,https://wikidata.org/entity/Q1205
249,https://example.org/db-delays/resource/Station...,Apolda railway station,https://wikidata.org/entity/Q800394,Apolda,http://www.wikidata.org/entity/Q490689,Thuringia,https://wikidata.org/entity/Q1205


## Q3 - Search stations by name and retrieve their detail-view identifiers.

In [10]:
query = """
PREFIX db: <https://example.org/db-delays/ontology/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?station ?stationName ?eva ?wikidata
WHERE {
  ?station a db:Station ; rdfs:label ?stationName ;
           db:evaNumber ?eva ; owl:sameAs ?wikidata .
  FILTER (CONTAINS(LCASE(STR(?stationName)), "berlin"))
}
ORDER BY ?stationName
"""

results = g.query(query)
display(pd.DataFrame([[str(v) if v is not None else None for v in row] for row in results],
                     columns=[str(v) for v in results.vars]))

,station,stationName,eva,wikidata
0,https://example.org/db-delays/resource/Station...,Berlin Albrechtshof railway station,8080040,https://wikidata.org/entity/Q800470
1,https://example.org/db-delays/resource/Station...,Berlin Alexanderplatz station,8011155,https://wikidata.org/entity/Q698497
2,https://example.org/db-delays/resource/Station...,Berlin Anhalter Bahnhof,8089002,https://wikidata.org/entity/Q695244
3,https://example.org/db-delays/resource/Station...,Berlin Bellevue station,8089005,https://wikidata.org/entity/Q374018
4,https://example.org/db-delays/resource/Station...,Berlin Bornholmer Straße station,8089008,https://wikidata.org/entity/Q32662318
5,https://example.org/db-delays/resource/Station...,Berlin Botanischer Garten station,8089009,https://wikidata.org/entity/Q800498
6,https://example.org/db-delays/resource/Station...,Berlin Frankfurter Allee station,8089013,https://wikidata.org/entity/Q552872
7,https://example.org/db-delays/resource/Station...,Berlin Greifswalder Straße station,8089011,https://wikidata.org/entity/Q439246
8,https://example.org/db-delays/resource/Station...,Berlin Nordbahnhof,8089024,https://wikidata.org/entity/Q673721
9,https://example.org/db-delays/resource/Station...,Berlin Ostbahnhof railway station,8010255,https://wikidata.org/entity/Q683239


## Q4 - Filter stations by region and train line.

In [ ]:
query = """
PREFIX db: <https://example.org/db-delays/ontology/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?station ?stationName ?stateName ?line ?wikidata
WHERE {
  VALUES ?state { <https://example.org/db-delays/resource/State/Berlin> }
  VALUES ?line { "3" }

  ?station a db:Station ;
           rdfs:label ?stationName ;
           db:inCity ?city ;
           owl:sameAs ?wikidata .

  ?city a db:City ;
        db:inState ?state .

  ?state a db:State ;
         rdfs:label ?stateName .

  ?stop a db:TrainStop ;
        db:atStation ?station ;
        db:lineLabel ?line .
}
ORDER BY ?stationName
"""

results = g.query(query)
display(pd.DataFrame([[str(v) if v is not None else None for v in row] for row in results],
                     columns=[str(v) for v in results.vars]))

,station,stationName,stateName,line,category,wikidata
0,https://example.org/db-delays/resource/Station...,Berlin Bellevue station,Berlin,3,4,https://wikidata.org/entity/Q374018
1,https://example.org/db-delays/resource/Station...,Berlin-Gesundbrunnen station,Berlin,3,1,https://wikidata.org/entity/Q678286
2,https://example.org/db-delays/resource/Station...,Berlin-Grunewald station,Berlin,3,4,https://wikidata.org/entity/Q457266
3,https://example.org/db-delays/resource/Station...,Berlin-Rummelsburg railway station,Berlin,3,4,https://wikidata.org/entity/Q15107132
4,https://example.org/db-delays/resource/Station...,Betriebsbahnhof Rummelsburg station,Berlin,3,4,https://wikidata.org/entity/Q829009


## Q5 - Map overview: coordinates and average arrival delay per linked station.

In [6]:
query = """
PREFIX db: <https://example.org/db-delays/ontology/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?station ?stationName ?latitude ?longitude ?wikidata
       (AVG(?delay) AS ?averageArrivalDelayMinutes)
       (COUNT(DISTINCT ?stop) AS ?measuredStops)
WHERE {
  ?station a db:Station ; rdfs:label ?stationName ;
           db:latitude ?latitude ; db:longitude ?longitude ;
           owl:sameAs ?wikidata .
  ?stop a db:TrainStop ; db:atStation ?station ; db:arrivalDelayMinutes ?delay .
}
GROUP BY ?station ?stationName ?latitude ?longitude ?wikidata
ORDER BY ?stationName
"""

results = g.query(query)
display(pd.DataFrame([[str(v) if v is not None else None for v in row] for row in results],
                     columns=[str(v) for v in results.vars]))

,station,stationName,latitude,longitude,wikidata,averageArrivalDelayMinutes,measuredStops
0,https://example.org/db-delays/resource/Station...,Aachen Central Station,50.7678,6.091499,https://wikidata.org/entity/Q300706,1.5,2
1,https://example.org/db-delays/resource/Station...,Aachen West station,50.78036,6.070715,https://wikidata.org/entity/Q320563,0,5
2,https://example.org/db-delays/resource/Station...,Aachen-Rothe Erde station,50.770202,6.116475,https://wikidata.org/entity/Q323235,0,1
3,https://example.org/db-delays/resource/Station...,Aalen Central station,48.841013,10.096271,https://wikidata.org/entity/Q319241,0,1
4,https://example.org/db-delays/resource/Station...,Achim railway station,53.01599,9.030447,https://wikidata.org/entity/Q15944978,49,1
...,...,...,...,...,...,...,...
246,https://example.org/db-delays/resource/Station...,Maintal West,50.140987,8.805296,https://wikidata.org/entity/Q65953232,0,2
247,https://example.org/db-delays/resource/Station...,Marienfelde station,52.423634,13.375127,https://wikidata.org/entity/Q800485,0.2,5
248,https://example.org/db-delays/resource/Station...,Nottuln-Appelhülsen station,51.892139,7.426334,https://wikidata.org/entity/Q15107241,0,2
249,https://example.org/db-delays/resource/Station...,Potsdam-Babelsberg station,52.391391,13.092871,https://wikidata.org/entity/Q801326,0.8333333333333333333333333333,6


## Q6 - Station details: daily recorded stops, lines and arrival punctuality.

In [7]:
query = """
PREFIX db: <https://example.org/db-delays/ontology/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?station ?stationName ?wikidata ?day
       (COUNT(DISTINCT ?stop) AS ?recordedStops)
       (GROUP_CONCAT(DISTINCT ?line; SEPARATOR=", ") AS ?lines)
       (COUNT(?delay) AS ?measuredArrivals)
       (AVG(?delay) AS ?averageArrivalDelayMinutes)
       (100.0 * AVG(?onTime) AS ?onTimePercent)
       (100.0 * AVG(?delayed) AS ?delayedPercent)
WHERE {
  ?station a db:Station ; rdfs:label ?stationName ;
           db:evaNumber "8089005" ; owl:sameAs ?wikidata .
  ?stop a db:TrainStop ; db:atStation ?station ; db:plannedArrival ?arrival .
  BIND (SUBSTR(STR(?arrival), 1, 10) AS ?day)
  OPTIONAL { ?stop db:lineLabel ?line }
  OPTIONAL {
    ?stop db:arrivalDelayMinutes ?delay .
    BIND (IF(?delay <= 0, 1, 0) AS ?onTime)
    BIND (IF(?delay > 0, 1, 0) AS ?delayed)
  }
}
GROUP BY ?station ?stationName ?wikidata ?day
ORDER BY ?day
"""

results = g.query(query)
display(pd.DataFrame([[str(v) if v is not None else None for v in row] for row in results],
                     columns=[str(v) for v in results.vars]))

,station,stationName,wikidata,day,recordedStops,lines,measuredArrivals,averageArrivalDelayMinutes,onTimePercent,delayedPercent
0,https://example.org/db-delays/resource/Station...,Berlin Bellevue station,https://wikidata.org/entity/Q374018,2024-07-08,24,"3, 7, 5, 9",24,0.125,87.5000,12.5000


## Q7 - Rank linked stations in a region by average arrival delay.

In [8]:
query = """
PREFIX db: <https://example.org/db-delays/ontology/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?station ?stationName ?stateName ?wikidata
       (AVG(?delay) AS ?averageArrivalDelayMinutes)
       (COUNT(DISTINCT ?stop) AS ?measuredStops)
WHERE {
  VALUES ?state { <https://example.org/db-delays/resource/State/Berlin> }
  ?station a db:Station ; rdfs:label ?stationName ;
           db:inCity ?city ; owl:sameAs ?wikidata .
  ?city a db:City ; db:inState <https://example.org/db-delays/resource/State/Berlin> .
  ?state a db:State ; rdfs:label ?stateName .
  ?stop a db:TrainStop ; db:atStation ?station ; db:arrivalDelayMinutes ?delay .
}
GROUP BY ?station ?stationName ?stateName ?wikidata
ORDER BY DESC(?averageArrivalDelayMinutes) ?stationName
"""

results = g.query(query)
display(pd.DataFrame([[str(v) if v is not None else None for v in row] for row in results],
                     columns=[str(v) for v in results.vars]))

,station,stationName,stateName,wikidata,averageArrivalDelayMinutes,measuredStops
0,https://example.org/db-delays/resource/Station...,Berlin-Charlottenburg railway station,Berlin,https://wikidata.org/entity/Q702596,2,3
1,https://example.org/db-delays/resource/Station...,Berlin Albrechtshof railway station,Berlin,https://wikidata.org/entity/Q800470,1.666666666666666666666666667,3
2,https://example.org/db-delays/resource/Station...,Berlin Ostbahnhof railway station,Berlin,https://wikidata.org/entity/Q683239,1.166666666666666666666666667,6
3,https://example.org/db-delays/resource/Station...,Berlin Zoologischer Garten railway station,Berlin,https://wikidata.org/entity/Q681785,1,5
4,https://example.org/db-delays/resource/Station...,Berlin-Buch station,Berlin,https://wikidata.org/entity/Q4892359,1,6
5,https://example.org/db-delays/resource/Station...,Berlin-Friedrichstraße railway station,Berlin,https://wikidata.org/entity/Q702402,1,4
6,https://example.org/db-delays/resource/Station...,Berlin-Gesundbrunnen station,Berlin,https://wikidata.org/entity/Q678286,1,1
7,https://example.org/db-delays/resource/Station...,Berlin-Karow railway station,Berlin,https://wikidata.org/entity/Q800480,1,7
8,https://example.org/db-delays/resource/Station...,Berlin-Spandau railway station,Berlin,https://wikidata.org/entity/Q654980,1,3
9,https://example.org/db-delays/resource/Station...,Berlin-Wannsee railway station,Berlin,https://wikidata.org/entity/Q567336,1,4
